# Olist Review Score Analysis: Phase 4 Multivariable Regression
Testing independent drivers of customer review scores across delivery delay, price, seller quality, category, and state.

In [1]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [2]:
# Load variables from .env
load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "5432")
dbname = os.getenv("DB_NAME", "olist")

engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}")

### 1. Load Dataset from SQL

In [3]:
query = Path("07_regression_dataset.sql").read_text()

In [4]:
df = pd.read_sql(query, engine)

In [5]:
df.head()
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 88699 entries, 0 to 88698
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id               88699 non-null  str    
 1   review_score           88699 non-null  int64  
 2   delay_days             86756 non-null  float64
 3   price                  88699 non-null  float64
 4   product_category_name  88699 non-null  str    
 5   seller_id              88699 non-null  str    
 6   customer_state         88699 non-null  str    
dtypes: float64(2), int64(1), str(4)
memory usage: 4.7 MB


order_id                    0
review_score                0
delay_days               1943
price                       0
product_category_name       0
seller_id                   0
customer_state              0
dtype: int64

### 2. Data Cleaning & Feature Engineering

In [6]:
df_model = df.dropna(subset=['delay_days'])
print(df_model.shape)

(86756, 7)


In [7]:
# 1. Calculate each seller's mean review score and map it to each row
seller_avg_score = df_model.groupby('seller_id')['review_score'].transform('mean')
df_model['seller_avg_score'] = seller_avg_score

In [8]:
# 2. Drop the raw text seller_id (we no longer need 3,000+ IDs)
df_model = df_model.drop(columns=['seller_id'])


In [9]:
df_encoded = pd.get_dummies(df_model, columns=['product_category_name', 'customer_state'], drop_first=True)
print(df_encoded.shape)

(86756, 104)


In [10]:
df_final = df_encoded.drop(columns=['order_id', 'seller_id'], errors='ignore')

### 3. Multivariable OLS Regression

In [11]:
import statsmodels.api as sm

# Separate features (X) and target outcome (y)
X = df_final.drop(columns=['review_score'])
y = df_final['review_score']

# Convert booleans (from dummy encoding) to 0.0/1.0 floats
X = X.astype(float)

# Add intercept column
X = sm.add_constant(X)

# Fit OLS regression
model = sm.OLS(y, X).fit()

# Print the report
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           review_score   R-squared:                       0.152
Model:                            OLS   Adj. R-squared:                  0.151
Method:                 Least Squares   F-statistic:                     152.2
Date:                Tue, 08 Sep 2026   Prob (F-statistic):               0.00
Time:                        01:04:04   Log-Likelihood:            -1.3401e+05
No. Observations:               86756   AIC:                         2.682e+05
Df Residuals:                   86653   BIC:                         2.692e+05
Df Model:                         102                                         
Covariance Type:            nonrobust                                         
                                                                           coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------